# Module 5 — Inverse Kinematics
## Unit 3 — Analytical (Closed-Form) Inverse Kinematics
### Lesson 3.3 — Decoupling Position and Orientation (Wrist-Partitioned Arms)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Wrist center pw = pd − d6 * (approach axis); position then orientation.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def ik_2link_closed(x, y, L1, L2):
    """Closed-form: return both (theta1,theta2); [] unreachable; one on boundary."""
    c2 = (x*x + y*y - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        s2 = sign*np.sqrt(max(0.0, 1.0 - c2*c2))
        t2 = np.arctan2(s2, c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(s2) < 1e-6:
            break
    return sols

def jacobian_2link(theta1, theta2, L1, L2):
    s1, s12 = np.sin(theta1), np.sin(theta1+theta2)
    c1, c12 = np.cos(theta1), np.cos(theta1+theta2)
    return np.array([[-L1*s1 - L2*s12, -L2*s12],
                     [ L1*c1 + L2*c12,  L2*c12]])

def wrist_center(p_d, R_d, d6):
    return np.asarray(p_d,float) - d6*np.asarray(R_d,float)[:,2]
R_up=np.eye(3)   # approach axis = z = (0,0,1)
print("pw =", wrist_center([0.5,0.2,0.3], R_up, 0.1))


## Coding Exercise (§8)
Wrist center for different approaches; R3_6 is a valid rotation.


In [ ]:
# YOUR CODE HERE
pw=wrist_center([0.5,0.2,0.3], R_up, 0.1)
assert np.allclose(pw,[0.5,0.2,0.2])
# sideways approach: rotate approach axis to +x
Rx=np.array([[0,0,1],[0,1,0],[-1,0,0.0]])  # maps z-> x in 3rd column
pw2=wrist_center([0.5,0.2,0.3], Rx, 0.1)
assert np.allclose(pw2,[0.4,0.2,0.3])
# decoupled orientation: R3_6=(R0_3)^-1 R_d is a rotation
R0_3=np.array([[0,-1,0],[1,0,0],[0,0,1.0]])
R3_6=np.linalg.inv(R0_3)@R_up
assert np.allclose(R3_6.T@R3_6,np.eye(3),atol=1e-9)
print("decoupling steps OK")


## Check your work


In [ ]:
assert np.allclose(wrist_center([0,0,0],np.eye(3),0.1),[0,0,-0.1])
print("All checks passed.")
